In [2]:
import pandas as pd
import tqdm
import numpy as np
import glob, os
from collections import defaultdict

def read_params(path):
    params = pd.read_csv(path, header=None, sep=" ", index_col=False, lineterminator='\n')
    params.rename(columns={0:'img_name'}, inplace=True)
    params = params.set_index('img_name').T.to_dict('list')
    return params

def swap_key(params):
    params_s = defaultdict(dict)
    for params_name, v in params.items():
        for img_name, params_value in v.items():
            params_s[img_name][params_name] = np.array(params_value).astype(np.float64)

    return params_s

def load_deca_params(deca_dir, cfg):
    deca_params = {}

    # face params 
    params_key = ['albedo', 'light', 'shape', 'exp', 'cam', 'detail', 'faceemb', 'pose', 'shadow', 'tform']
    for k in tqdm.tqdm(params_key, desc="Loading deca params..."):
        params_path = glob.glob(f"{deca_dir}/*{k}-anno.txt")
        for path in params_path:
            deca_params[k] = read_params(path=path)
    
    deca_params = swap_key(deca_params)
    return deca_params

deca_dir = '/data/mint/DPM_Dataset/ffhq_256_with_anno/params/valid/'
deca_params = load_deca_params(deca_dir, cfg=None)
print(deca_params.keys())

Loading deca params...: 100%|██████████| 10/10 [00:06<00:00,  1.56it/s]


dict_keys(['60000.jpg', '60001.jpg', '60002.jpg', '60003.jpg', '60004.jpg', '60005.jpg', '60006.jpg', '60007.jpg', '60008.jpg', '60009.jpg', '60010.jpg', '60011.jpg', '60012.jpg', '60013.jpg', '60014.jpg', '60015.jpg', '60016.jpg', '60017.jpg', '60018.jpg', '60019.jpg', '60020.jpg', '60021.jpg', '60022.jpg', '60023.jpg', '60024.jpg', '60025.jpg', '60026.jpg', '60027.jpg', '60028.jpg', '60029.jpg', '60030.jpg', '60031.jpg', '60032.jpg', '60033.jpg', '60034.jpg', '60035.jpg', '60036.jpg', '60037.jpg', '60038.jpg', '60039.jpg', '60040.jpg', '60041.jpg', '60042.jpg', '60043.jpg', '60044.jpg', '60045.jpg', '60046.jpg', '60047.jpg', '60048.jpg', '60049.jpg', '60050.jpg', '60051.jpg', '60052.jpg', '60053.jpg', '60054.jpg', '60055.jpg', '60056.jpg', '60057.jpg', '60058.jpg', '60059.jpg', '60060.jpg', '60061.jpg', '60062.jpg', '60063.jpg', '60064.jpg', '60065.jpg', '60066.jpg', '60067.jpg', '60068.jpg', '60069.jpg', '60070.jpg', '60071.jpg', '60072.jpg', '60073.jpg', '60074.jpg', '60075.jpg', '

In [11]:
to_copy = ['image', 'params', 'face_segment', 'shadow_map', 'rendered_image']
params = ['albedo', 'light', 'shape', 'exp', 'cam', 'detail', 'faceemb', 'pose', 'shadow', 'tform']

img_name = '60065.jpg'
for tc in to_copy:
    print(f"[#] Adding {tc}...")
    if tc == 'image':
        os.system(f"cp /data/mint/DPM_Dataset/ffhq_256_with_anno/ffhq_256/valid/{img_name} /data/mint/DPM_Dataset/Hard_CS/images_aligned/valid/{img_name}")
    elif tc == 'face_segment':
        os.system(f"cp /data/mint/DPM_Dataset/ffhq_256_with_anno/face_segment_with_pupil/valid/anno/anno_{img_name.replace('.jpg', '.png')} /data/mint/DPM_Dataset/Hard_CS/face_segment_with_pupil/valid/anno/")
    elif tc == 'shadow_map':
        os.system(f"cp /data/mint/DPM_Dataset/ffhq_256_with_anno/shadow_diff_SS_with_c_simplified/valid/{img_name.replace('.jpg', '.npy')} /data/mint/DPM_Dataset/Hard_CS/shadow_diff_SS_with_c_simplified/valid/")
    elif tc == 'rendered_image':
        os.system(f"cp /data/mint/DPM_Dataset/ffhq_256_with_anno/rendered_images/deca_masked_face_images_woclip/valid/{img_name.replace('.jpg', '.npy')} /data/mint/DPM_Dataset/Hard_CS/rendered_images/deca_masked_face_images_woclip/valid/")
    elif tc == 'params':
        # Load params from DECA first
        deca_params_sj = deca_params[img_name]
        for p in deca_params_sj.keys():
            # Load the params from Hard_CS
            with open(f"/data/mint/DPM_Dataset/Hard_CS/params/valid/ffhq-valid-{p}-anno.txt", "a+") as f:
                append_text = img_name + " " + " ".join([str(x) for x in deca_params_sj[p]]) + "\n"
                f.write(append_text)
    

[#] Adding image...
[#] Adding params...
[#] Adding face_segment...
[#] Adding shadow_map...
[#] Adding rendered_image...
